# Track1039: Sailor2-20B classification and Qwen2.5-14B correction

Attach the PRIVATE **VGEC26-track1039-private-20261019.zip** package alone. Enable Internet and **GPU T4 x2**. Keep repository-read GITHUB_TOKEN; HF_TOKEN is optional, and WANDB_API_KEY enables monitoring. Save Version / Run All. Dataset/extraction names may vary. No notebook toggles or older output dataset are needed.

The committed recipe performs two fresh **NF4 QLoRA** fits, FP16 compute, five FULL epochs each, retaining the best checkpoints. It compares against the bundled winning October19 Sailor7B classifier and old Qwen7B greedy corrector, selecting each component independently. Seed42, same78 development rows, unchanged anchored training data, private2034 inference only, no folds/hints, greedy decoding. Large frozen bases use a temporary one-base cache and may redownload between stages; keep about40GiB free for cache/download space. Root W&B run stays alive through final export. This native recipe requires two GPUs; it cannot fit the local8GB RTX4060.

Download **results.zip** and retain the whole run with selected_models/ and selected.json. Use selected.json with tools.replay_focused for exact replay. The quality-data track1038 experiment is preserved at **archive/track1038-quality-20261019**, including its source/docs/private backup; this1039 recipe does not modify those artifacts. Development scores and memory estimates do not guarantee an official score or native runtime success.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
SOURCE_ROOT = Path("/kaggle/working")
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Fetch the committed source securely into a fresh directory.
from urllib.parse import urlsplit
import tempfile

with tempfile.TemporaryDirectory() as authentication:
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    token = None
    if urlsplit(REPO_URL).hostname == "github.com":
        from kaggle_secrets import UserSecretsClient
        try:
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            raise RuntimeError("Enable GITHUB_TOKEN in Kaggle Secrets with repository read access.") from None
        if not token:
            raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
        askpass = Path(authentication) / "git-askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                           '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)
    PROJECT_DIR = Path(tempfile.mkdtemp(prefix="gecc-source-", dir=SOURCE_ROOT))
    result = subprocess.run(["git", "-c", "credential.helper=", "clone", "--depth", "1",
                             "--single-branch", "--branch", BRANCH, "--", REPO_URL, str(PROJECT_DIR)],
                            env=env, capture_output=True, text=True)
    returncode = result.returncode
    detail = result.stderr.replace(token, "[REDACTED]") if token else result.stderr
    del env, token, result
    if returncode:
        raise RuntimeError("Cannot clone source; check Internet, branch, and GITHUB_TOKEN access. " + detail.strip())
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("gecc_notebook", PROJECT_DIR / "tools/notebook.py")
launcher = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher)
TRAINING_PYTHON = launcher.setup_kaggle(PROJECT_DIR)


In [ ]:
RUN_DIR = launcher.launch(PROJECT_DIR, TRAINING_PYTHON)
